# `project_kaggle_falcon_7b` — MIND-style hallucination detection with drift + variance + entropy

**Model:** `tiiuae/falcon-7b`
**Target compute:** Kaggle Free (T4×2 bf16)
**Samples / class:** 1000 (Wikipedia continuation, MIND Algorithm 1)

Pipeline:
1. Load LLM (bf16, model-parallel across all available GPUs).
2. Generate hallucinated + non-hallucinated Wikipedia continuations.
3. Per sample, extract:
   - **canonical**: last-token last-layer hidden state (D-dimensional)
   - **D_mean**: mean cosine distance between adjacent layers
   - **V_last**: L2 variance of last-token activations across layers
   - **H_mean**: mean Shannon entropy of per-step token distribution
4. **Save the full generated dataset** to `kaggle_falcon_7b_dataset_full.json` (BLOCK 6.5 — new vs. the Falcon-7B Kaggle build).
5. Train a 5-layer MLP on `[canonical || D_mean || V_last || H_mean]` (StandardScaler on the 3 scalars).
6. Evaluate on Wikipedia held-out + 7 downstream datasets (TruthfulQA, TriviaQA, CoQA, TydiQA-GP, HaluEval-{QA, Summ, Dialog}).
7. Write `kaggle_falcon_7b_results.json` (HalluShift-style + smoke-style diagnostics) for downstream comparison.

**Output files** dropped in the working directory:
| File | Purpose |
|---|---|
| `kaggle_falcon_7b_dataset_full.json` | every generated record (pre-split) — for ablations |
| `kaggle_falcon_7b_train.json` / `_test.json` | 80/20 split used for the MLP |
| `kaggle_falcon_7b_checkpoint.json` | autosave every 200 samples (resume after disconnect) |
| `kaggle_falcon_7b_mind_plus_best.pth` | best MLP weights + scaler tensors |
| `kaggle_falcon_7b_results.json` | env/config/sanity/metrics/timings — single file for review |


In [ ]:
# =============================================================================
# BLOCK 0: PIP INSTALLS
# =============================================================================
!pip install -q -U "transformers>=4.45" "tokenizers>=0.19" "accelerate>=0.30"
!pip install -q -U datasets spacy nltk scikit-learn tqdm sentence-transformers
!pip install -q -U bitsandbytes
!python -m spacy download en_core_web_sm


In [ ]:
# =============================================================================
# BLOCK 1: CONFIG + IMPORTS + MODEL LOADING + DIAGNOSTICS BOOTSTRAP
# =============================================================================
import os, gc, json, random, math, time, datetime, platform, sys, traceback
import numpy as np
import torch
import torch.nn.functional as F
import spacy, nltk
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from nltk.tokenize import sent_tokenize
from sklearn.preprocessing import StandardScaler

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
MODEL_NAME      = "tiiuae/falcon-7b"
TARGET_SAMPLES  = 1000
TOPK_FIRST_TOKEN = 4
WINDOWS         = 16
SEED            = 42
USE_4BIT        = False
DTYPE           = torch.bfloat16 if torch.cuda.is_available() else torch.float32
MAX_GEN_NEW     = 48
MODEL_TAG       = "kaggle_falcon_7b"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------------------------------------------------------------
# DIAGNOSTICS DICT — accumulated across the whole notebook and dumped at end
# -----------------------------------------------------------------------------
DIAG = {
    "stage_timings_sec": {},
    "env": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
        "cuda_capability": (torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None),
        "free_vram_gb": (round(torch.cuda.mem_get_info()[0]/1e9, 2) if torch.cuda.is_available() else None),
    },
    "config": {
        "model_name": MODEL_NAME, "model_tag": MODEL_TAG,
        "target_samples_per_class": TARGET_SAMPLES,
        "topk_first_token": TOPK_FIRST_TOKEN, "windows": WINDOWS,
        "seed": SEED, "use_4bit": USE_4BIT, "dtype": str(DTYPE),
        "max_gen_new_tokens": MAX_GEN_NEW,
        "target_env": "kaggle",
    },
    "library_versions": {},
    "model_info": {},
    "data_gen": {},
    "feature_sanity": {},
    "wikipedia_eval": {},
    "downstream_datasets": {},
    "multitask": {},
    "errors": [],
}

for _mod_name in ["transformers", "datasets", "accelerate", "sklearn",
                  "spacy", "nltk", "numpy", "tqdm", "sentence_transformers"]:
    try:
        _m = __import__(_mod_name)
        DIAG["library_versions"][_mod_name] = getattr(_m, "__version__", "n/a")
    except Exception as _e:
        DIAG["library_versions"][_mod_name] = f"IMPORT_FAILED: {_e}"

# -----------------------------------------------------------------------------
# Tokeniser + model
# -----------------------------------------------------------------------------
print("=" * 80); print("BLOCK 1: MODEL LOADING"); print("=" * 80)
print(f"Loading {MODEL_NAME} (4bit={USE_4BIT}, dtype={DTYPE}) ...")

_t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

load_kwargs = dict(trust_remote_code=False, torch_dtype=DTYPE,
                   device_map="auto" if torch.cuda.is_available() else None)
if USE_4BIT and torch.cuda.is_available():
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=DTYPE, bnb_4bit_quant_type="nf4")

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()
for p in model.parameters():
    p.requires_grad = False
DIAG["stage_timings_sec"]["model_load"] = round(time.perf_counter() - _t0, 2)

nlp = spacy.load("en_core_web_sm")

DIAG["model_info"] = {
    "hidden_dim":    int(model.config.hidden_size),
    "n_layers":      int(model.config.num_hidden_layers),
    "n_heads":       int(getattr(model.config, "num_attention_heads", -1)),
    "vocab_size":    int(model.config.vocab_size),
    "max_pos_embed": int(getattr(model.config, "max_position_embeddings", -1)),
    "param_count_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
    "device":        str(next(model.parameters()).device),
}

if torch.cuda.is_available():
    print(f"✓ GPU              : {torch.cuda.get_device_name(0)}")
    print(f"✓ Free VRAM        : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"✓ hidden_dim       : {DIAG['model_info']['hidden_dim']}")
print(f"✓ n_layers         : {DIAG['model_info']['n_layers']}")
print(f"✓ vocab_size       : {DIAG['model_info']['vocab_size']}")
print(f"✓ max_pos_embed    : {DIAG['model_info']['max_pos_embed']}")
print(f"✓ param_count (M)  : {DIAG['model_info']['param_count_M']}")
print(f"✓ device           : {DIAG['model_info']['device']}")
print(f"✓ model_load time  : {DIAG['stage_timings_sec']['model_load']} s")
print(f"✓ Model tag        : {MODEL_TAG}")


In [ ]:
# =============================================================================
# BLOCK 2: ENTITY EXTRACTION (spaCy NER + post-processing)
# =============================================================================
def delete_substrings(lst):
    substrings = []
    lst = list(set(lst))
    for s in lst:
        if any(s in o for o in lst if o != s):
            substrings.append(s)
    for s in substrings:
        lst.remove(s)
    return lst


def find_boundaries(text, words):
    boundaries = []
    for word in words:
        ntext = text
        while True:
            start = ntext.find(word)
            if start == -1: break
            end = start + len(word) - 1
            while start > 0 and ntext[start-1] != " ": start -= 1
            while end < len(ntext)-1 and ntext[end+1] != " ": end += 1
            boundaries.append("".join(ntext[i] for i in range(start, end+1)))
            ntext = ntext[end+1:]
    return boundaries


def get_entities(text):
    ents = list({str(e) for e in nlp(text).ents})
    ents = find_boundaries(text, ents)
    ents = delete_substrings(ents)
    out = []
    for i in range(len(text)):
        for e in ents:
            if text[i:].startswith(e):
                out.append((e, i))
    return out


print("✓ entity extractor defined")


In [ ]:
# =============================================================================
# BLOCK 3: TOKEN-BOUNDARY FINDER (MIND Algorithm 1, step 2)
# =============================================================================
def find_first_and_next_token(text, e, idx, input_id, prompt=""):
    new_text = f"{text[:idx].strip()} {text[idx:].replace(e, e+' @', 1).strip()}"
    new_input_id = tokenizer(prompt + new_text.strip(), return_tensors="pt")["input_ids"].tolist()[0]
    for i in range(len(input_id[0])):
        if input_id[0][i] != new_input_id[i]:
            return []
    first_token = new_input_id[len(input_id[0])]
    at_cands = tokenizer("@", add_special_tokens=False)["input_ids"]
    at_cands += tokenizer(" @", add_special_tokens=False)["input_ids"]
    at_pos = None
    for at_tok in at_cands:
        try:
            at_pos = new_input_id.index(at_tok, len(input_id[0]))
            break
        except ValueError: continue
    if at_pos is None or at_pos >= len(new_input_id) - 1:
        return []
    next_token = new_input_id[at_pos + 1]
    entity_len = at_pos - len(input_id[0])
    last_id = new_input_id[at_pos + 1:]
    return [first_token, next_token, entity_len, last_id]


print("✓ token finder defined")


In [ ]:
# =============================================================================
# BLOCK 4: FEATURE EXTRACTION  (canonical MIND + D_mean + V_last + H_last)
# =============================================================================
@torch.no_grad()
def extract_features(text: str):
    enc = tokenizer(text.strip(), return_tensors="pt",
                    truncation=True,
                    max_length=getattr(model.config, "max_position_embeddings", 4096)).to(model.device)
    out = model(**enc, output_hidden_states=True, use_cache=False)
    hs = out.hidden_states
    last_token_per_layer = torch.stack(
        [h[0, -1, :].float() for h in hs[1:]], dim=0
    )
    canonical = last_token_per_layer[-1].cpu().tolist()

    if last_token_per_layer.shape[0] < 2:
        D_mean = 0.0
    else:
        a = last_token_per_layer[:-1]
        b = last_token_per_layer[1:]
        cos = F.cosine_similarity(a, b, dim=1)
        D_mean = float((1.0 - cos).mean().item())

    mean_h = last_token_per_layer.mean(dim=0)
    V_last = float(((last_token_per_layer - mean_h) ** 2).sum(dim=1).mean().item())

    last_logits = out.logits[0, -1, :].float()
    p_last = torch.softmax(last_logits, dim=-1)
    H_last = float(-(p_last * p_last.clamp_min(1e-12).log()).sum().item())

    return canonical, D_mean, V_last, H_last


print("✓ extract_features defined (canonical, D_mean, V_last, H_last)")


In [ ]:
# =============================================================================
# BLOCK 5: SANITY TEST (records intermediate values to DIAG["feature_sanity"])
# =============================================================================
sample_text = ("Albert Einstein was born in Ulm, in the Kingdom of Württemberg in the "
               "German Empire, on 14 March 1879. His parents were Hermann Einstein, "
               "a salesman and engineer, and Pauline Koch.")

ents = get_entities(sample_text)
title = "Albert Einstein"
valid = [(e, i) for e, i in ents if i != 0 and e.lower() not in title.lower()]
DIAG["feature_sanity"]["entity_count_total"] = len(ents)
DIAG["feature_sanity"]["entity_count_valid"] = len(valid)
DIAG["feature_sanity"]["entity_examples"]    = [{"entity": e, "char_idx": i} for e, i in ents[:8]]

print(f"✓ entity occurrences: {len(ents)}")
print(f"✓ valid entities    : {len(valid)}")

if valid:
    canon, D_mean, V_last, H_last = extract_features(sample_text)
    DIAG["feature_sanity"]["canonical_dim"]   = len(canon)
    DIAG["feature_sanity"]["canonical_first_8"] = [round(float(x), 5) for x in canon[:8]]
    DIAG["feature_sanity"]["canonical_norm"]  = float(np.linalg.norm(np.asarray(canon, dtype=np.float32)))
    DIAG["feature_sanity"]["D_mean"]          = D_mean
    DIAG["feature_sanity"]["V_last"]          = V_last
    DIAG["feature_sanity"]["H_last"]          = H_last

    print(f"✓ canonical dim     : {len(canon)}")
    print(f"✓ canonical norm    : {DIAG['feature_sanity']['canonical_norm']:.3f}")
    print(f"✓ D_mean            : {D_mean:.4f}")
    print(f"✓ V_last            : {V_last:.4f}")
    print(f"✓ H_last            : {H_last:.4f}")

    assert len(canon) == model.config.hidden_size, "canonical dim mismatch"
    assert 0.0 <= D_mean <= 2.0, f"D_mean out of range: {D_mean}"
    assert V_last > 0.0, f"V_last must be > 0: {V_last}"
    assert 0.0 <= H_last <= math.log(model.config.vocab_size) + 1e-3, \
        f"H_last out of range: {H_last}"
    print("✓ all sanity asserts passed")
    DIAG["feature_sanity"]["asserts_passed"] = True
else:
    DIAG["feature_sanity"]["asserts_passed"] = False
    DIAG["errors"].append("sanity_test: no valid entities found in Einstein sample")


In [ ]:
# =============================================================================
# BLOCK 6: DATA GENERATION  (MIND Algorithm 1, with H_mean computed during gen)
# =============================================================================
def per_step_entropy(scores):
    if not scores:
        return 0.0
    Hs = []
    for s in scores:
        p = torch.softmax(s[0].float(), dim=-1)
        H = -(p * (p.clamp_min(1e-12)).log()).sum().item()
        Hs.append(H)
    return float(sum(Hs) / len(Hs))


SKIP_COUNTERS = {"overflow_prefix": 0, "no_token_match": 0, "overflow_suffix": 0,
                 "model_knew": 0, "no_match_in_topk": 0, "empty_entity": 0,
                 "duplicate_entity": 0, "exception": 0}


def generate_sample(text, entity, idx, title):
    MAX_POS = getattr(model.config, "max_position_embeddings", 4096)
    HEADROOM = WINDOWS + 64
    enc = tokenizer(text[:idx].strip(), return_tensors="pt").to(model.device)
    input_ids = enc["input_ids"]; attn = enc["attention_mask"]
    if input_ids.shape[1] + HEADROOM >= MAX_POS:
        SKIP_COUNTERS["overflow_prefix"] += 1
        return None
    input_id_list = input_ids.tolist()
    toks = find_first_and_next_token(text, entity, idx, input_id_list)
    if not toks:
        SKIP_COUNTERS["no_token_match"] += 1
        return None
    first_, next_, entity_len, last_id = toks
    if input_ids.shape[1] + entity_len + len(last_id) + HEADROOM >= MAX_POS:
        SKIP_COUNTERS["overflow_suffix"] += 1
        return None

    gen = model.generate(input_ids, attention_mask=attn,
                         max_new_tokens=entity_len + WINDOWS,
                         return_dict_in_generate=True, output_scores=True,
                         do_sample=False, pad_token_id=tokenizer.eos_token_id)
    if first_ in torch.topk(gen.scores[0], k=TOPK_FIRST_TOKEN).indices[0].tolist():
        SKIP_COUNTERS["model_knew"] += 1
        return None
    found_step = None
    for step, sc in enumerate(gen.scores):
        if next_ in torch.topk(sc, k=TOPK_FIRST_TOKEN).indices[0].tolist():
            found_step = step; break
    if found_step is None:
        SKIP_COUNTERS["no_match_in_topk"] += 1
        return None

    H_mean_hall = per_step_entropy(gen.scores[: found_step + 1])
    new_seq = gen.sequences[0, : input_ids.shape[1] + found_step].tolist()
    new_entity_ids = new_seq[input_ids.shape[1]:]
    hallucinated_ids = input_id_list[0] + new_entity_ids + last_id
    hallucinated_text = tokenizer.decode(hallucinated_ids, skip_special_tokens=True)
    new_entity = tokenizer.decode(new_entity_ids, skip_special_tokens=True).strip().lower()
    if not new_entity:
        SKIP_COUNTERS["empty_entity"] += 1
        return None
    if entity.lower() in new_entity or new_entity in text.lower():
        SKIP_COUNTERS["duplicate_entity"] += 1
        return None

    canon_h, D_h, V_h, _ = extract_features(hallucinated_text)
    canon_o, D_o, V_o, H_o = extract_features(text)

    return {
        "text_hall": hallucinated_text, "entity_hall": new_entity,
        "canon_h": canon_h, "D_h": D_h, "V_h": V_h, "H_h": H_mean_hall,
        "text_orig": text, "entity_orig": entity,
        "canon_o": canon_o, "D_o": D_o, "V_o": V_o, "H_o": H_o,
        "title": title,
    }


print("\nLoading Wikipedia (streaming) ...")
wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

dataset_hall, dataset_non_hall = [], []
processed = 0
print(f"Generating up to {TARGET_SAMPLES*2} samples ({TARGET_SAMPLES}/class)\n")
pbar = tqdm(total=TARGET_SAMPLES * 2, desc="samples")
_t0 = time.perf_counter()
for row in wiki:
    if len(dataset_hall) >= TARGET_SAMPLES and len(dataset_non_hall) >= TARGET_SAMPLES:
        break
    processed += 1
    try:
        sents = sent_tokenize(row["text"])
        if len(sents) < 2: continue
        text = " ".join(sents[:2])
        title = row.get("title", "")
        ents = get_entities(text)
        if not ents: continue
        seen, ents_filt = set(), []
        for e, i in ents:
            if i == 0 or e.lower() in title.lower(): continue
            if i not in seen:
                seen.add(i); ents_filt.append((e, i))
        if not ents_filt: continue
        entity, char_idx = random.choice(ents_filt)
        result = generate_sample(text, entity, char_idx, title)
        if result is None: continue

        if len(dataset_hall) < TARGET_SAMPLES:
            dataset_hall.append({
                "label": 1, "text": result["text_hall"], "entity": result["entity_hall"],
                "embedding": result["canon_h"], "D_mean": result["D_h"],
                "V_last": result["V_h"], "H_mean": result["H_h"], "title": result["title"],
            })
            pbar.update(1)
        if len(dataset_non_hall) < TARGET_SAMPLES:
            dataset_non_hall.append({
                "label": 0, "text": result["text_orig"], "entity": result["entity_orig"],
                "embedding": result["canon_o"], "D_mean": result["D_o"],
                "V_last": result["V_o"], "H_mean": result["H_o"], "title": result["title"],
            })
            pbar.update(1)
        pbar.set_postfix(H1=len(dataset_hall), H0=len(dataset_non_hall), proc=processed)
        # Incremental checkpoint every 200 samples so a session disconnect
        # doesn't waste all the GPU time.
        if (len(dataset_hall) + len(dataset_non_hall)) % 200 == 0:
            with open(f"{MODEL_TAG}_checkpoint.json", "w") as f:
                json.dump(dataset_hall + dataset_non_hall, f)
    except Exception as e:
        SKIP_COUNTERS["exception"] += 1
        if processed % 1000 == 0:
            print(f"\n[warn at {processed}]: {e}")
        continue
    if processed % 100 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()

pbar.close()
DIAG["stage_timings_sec"]["data_gen"] = round(time.perf_counter() - _t0, 2)
DIAG["data_gen"] = {
    "wiki_articles_processed": processed,
    "n_hallucinated": len(dataset_hall),
    "n_non_hallucinated": len(dataset_non_hall),
    "skip_counters": dict(SKIP_COUNTERS),
    "example_hall": (dataset_hall[0]["text"][:280] if dataset_hall else None),
    "example_non_hall": (dataset_non_hall[0]["text"][:280] if dataset_non_hall else None),
    "example_hall_entity":    (dataset_hall[0]["entity"] if dataset_hall else None),
    "example_non_hall_entity":(dataset_non_hall[0]["entity"] if dataset_non_hall else None),
}
print(f"\n✓ done. H=1: {len(dataset_hall)}  H=0: {len(dataset_non_hall)}")
print(f"✓ data_gen time: {DIAG['stage_timings_sec']['data_gen']} s")
print(f"✓ skip_counters: {SKIP_COUNTERS}")


In [ ]:
# =============================================================================
# BLOCK 6.5: SAVE FULL GENERATED DATASET   (NEW — added 2026-05-28)
# Persists every generated record (both classes, pre-split) to one JSON.
# Useful for offline ablations, re-running the classifier with different
# feature subsets, or auditing the data without re-running the LLM.
# =============================================================================
DATASET_FULL_PATH = f"{MODEL_TAG}_dataset_full.json"
_full_dataset_records = dataset_hall + dataset_non_hall
with open(DATASET_FULL_PATH, "w") as _f:
    json.dump(_full_dataset_records, _f)
DIAG["data_gen"]["dataset_full_path"]      = DATASET_FULL_PATH
DIAG["data_gen"]["dataset_full_n_records"] = len(_full_dataset_records)
DIAG["data_gen"]["dataset_full_size_mb"]   = round(os.path.getsize(DATASET_FULL_PATH) / 1e6, 2)
print(f"✓ wrote {DATASET_FULL_PATH}  ({DIAG['data_gen']['dataset_full_size_mb']} MB, "
      f"{len(_full_dataset_records)} records)")


In [ ]:
# =============================================================================
# BLOCK 7: TRAIN / TEST SPLIT + SAVE
# =============================================================================
dataset = dataset_hall + dataset_non_hall
random.shuffle(dataset)
split = int(0.8 * len(dataset))
train_data, test_data = dataset[:split], dataset[split:]
TRAIN_PATH = f"{MODEL_TAG}_train.json"
TEST_PATH  = f"{MODEL_TAG}_test.json"
with open(TRAIN_PATH, "w") as f: json.dump(train_data, f)
with open(TEST_PATH,  "w") as f: json.dump(test_data,  f)
DIAG["data_gen"]["train_size"] = len(train_data)
DIAG["data_gen"]["test_size"]  = len(test_data)
print(f"✓ train: {len(train_data)} -> {TRAIN_PATH}")
print(f"✓ test:  {len(test_data)} -> {TEST_PATH}")


In [ ]:
# =============================================================================
# BLOCK 8: MLP CLASSIFIER  (input = [canonical || z(D_mean) || z(V_last) || z(H_mean)])
# =============================================================================
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              roc_auc_score, confusion_matrix, classification_report,
                              brier_score_loss)


class MINDPlusClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 2),
        )
    def forward(self, x):
        return self.layers(x)


def build_feature_matrix(records, scaler=None, fit=False):
    canon = np.array([r["embedding"] for r in records], dtype=np.float32)
    scalars = np.array([[r["D_mean"], r["V_last"], r["H_mean"]] for r in records],
                       dtype=np.float32)
    if fit:
        scaler = StandardScaler().fit(scalars)
    scalars_z = scaler.transform(scalars).astype(np.float32)
    X = np.concatenate([canon, scalars_z], axis=1)
    y = np.array([r["label"] for r in records], dtype=np.int64)
    return X, y, scaler


class FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


print("✓ MLP + scaler helpers defined")


In [ ]:
# =============================================================================
# BLOCK 9: TRAIN MLP ON COMBINED FEATURES
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_tr, y_tr, scaler = build_feature_matrix(train_data, fit=True)
X_te, y_te, _      = build_feature_matrix(test_data,  scaler=scaler, fit=False)
input_dim = X_tr.shape[1]
print(f"✓ feature dim: {input_dim}  (= {input_dim-3} embedding + 3 scalars)")

DIAG["model_info"]["mlp_input_dim"] = int(input_dim)
DIAG["model_info"]["mlp_arch"]      = "5-layer MLP: 512 → 256 → 128 → 64 → 2"

mlp = MINDPlusClassifier(input_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(mlp.parameters(), lr=5e-4, weight_decay=1e-5)
train_loader = DataLoader(FeatureDataset(X_tr, y_tr), batch_size=32, shuffle=True)
test_loader  = DataLoader(FeatureDataset(X_te, y_te), batch_size=32, shuffle=False)

EPOCHS = 10; best_acc = 0.0
epoch_history = []
_t0 = time.perf_counter()
for ep in range(EPOCHS):
    mlp.train()
    epoch_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); logits = mlp(x); l = loss_fn(logits, y); l.backward(); opt.step()
        epoch_loss += float(l.item())
    mlp.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (mlp(x).argmax(1) == y).sum().item(); total += y.size(0)
    acc_ep = correct / max(total, 1)
    epoch_history.append({"epoch": ep+1,
                          "train_loss": round(epoch_loss / max(len(train_loader),1), 4),
                          "test_acc": round(acc_ep, 4)})
    star = " ★" if acc_ep > best_acc else ""
    if acc_ep > best_acc:
        best_acc = acc_ep
        torch.save({"model_state": mlp.state_dict(), "scaler_mean": scaler.mean_,
                    "scaler_scale": scaler.scale_, "input_dim": input_dim},
                   f"{MODEL_TAG}_mind_plus_best.pth")
    print(f"epoch {ep+1:2d}/{EPOCHS}  test_acc={acc_ep:.4f}{star}")
DIAG["stage_timings_sec"]["mlp_train"] = round(time.perf_counter() - _t0, 2)
DIAG["wikipedia_eval"]["mlp_epoch_history"] = epoch_history
DIAG["wikipedia_eval"]["mlp_best_acc"] = float(best_acc)
print(f"✓ best test acc: {best_acc:.4f}")
print(f"✓ mlp_train time: {DIAG['stage_timings_sec']['mlp_train']} s")


In [ ]:
# =============================================================================
# BLOCK 10: WIKIPEDIA HELD-OUT EVALUATION (Acc/Prec/Rec/F1/AUROC/Brier)
# =============================================================================
_ckpt = torch.load(f"{MODEL_TAG}_mind_plus_best.pth", weights_only=False)
mlp.load_state_dict(_ckpt["model_state"])
mlp.eval()
try:
    _ = scaler.mean_
except (NameError, AttributeError):
    from sklearn.preprocessing import StandardScaler as _SS
    scaler = _SS()
    scaler.mean_  = np.asarray(_ckpt["scaler_mean"],  dtype=np.float64)
    scaler.scale_ = np.asarray(_ckpt["scaler_scale"], dtype=np.float64)
    scaler.var_   = scaler.scale_ ** 2
    scaler.n_features_in_ = len(scaler.mean_)
all_p, all_y, all_pr = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = mlp(x)
        prob = torch.softmax(logits, dim=1)[:, 1]
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_y.extend(y.numpy())
        all_pr.extend(prob.cpu().numpy())
all_p = np.array(all_p); all_y = np.array(all_y); all_pr = np.array(all_pr)

def _safe(fn, *a, **kw):
    try:    return float(fn(*a, **kw))
    except Exception as e:
        DIAG["errors"].append(f"wikipedia_eval metric: {fn.__name__}: {e}")
        return float("nan")

acc  = _safe(accuracy_score, all_y, all_p)
prec, rec, f1, _ = precision_recall_fscore_support(all_y, all_p, average="binary",
                                                     zero_division=0)
auc   = _safe(roc_auc_score, all_y, all_pr) if len(set(all_y.tolist())) > 1 else float("nan")
brier = _safe(brier_score_loss, all_y, all_pr)
cm    = confusion_matrix(all_y, all_p, labels=[0,1])

print(f"\nWikipedia held-out test ({len(all_y)} samples):")
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1       : {f1:.4f}")
print(f"  AUROC    : {auc:.4f}")
print(f"  Brier    : {brier:.4f}")

wiki_metrics = {"accuracy": float(acc), "precision": float(prec), "recall": float(rec),
                "f1": float(f1), "auc_roc": float(auc), "brier": float(brier),
                "n": int(len(all_y)),
                "cm": {"tn": int(cm[0,0]), "fp": int(cm[0,1]),
                       "fn": int(cm[1,0]), "tp": int(cm[1,1])},
                "label_distribution": {"y=0": int((all_y==0).sum()),
                                       "y=1": int((all_y==1).sum())},
                "pred_distribution":  {"p=0": int((all_p==0).sum()),
                                       "p=1": int((all_p==1).sum())},
                "prob_min":  round(float(all_pr.min()) if len(all_pr) else 0.0, 4),
                "prob_max":  round(float(all_pr.max()) if len(all_pr) else 0.0, 4),
                "prob_mean": round(float(all_pr.mean()) if len(all_pr) else 0.0, 4)}
DIAG["wikipedia_eval"].update(wiki_metrics)


In [ ]:
# =============================================================================
# BLOCK 11: DOWNLOAD 7 MULTI-TASK EVAL DATASETS
# Uses namespaced HF IDs with legacy-ID fallback (fixes the Falcon-7B run).
# =============================================================================
print("Loading the 7 datasets ...")
def safe_load_first(*tries, subsample=None, label=""):
    # Try loaders in order; return the first that succeeds.
    last_err = None
    for i, loader_fn in enumerate(tries):
        try:
            ds = loader_fn()
            if subsample and len(ds) > subsample:
                ds = ds.shuffle(seed=SEED).select(range(subsample))
            print(f"  ✓ {label}: {len(ds)} samples  (via loader #{i})")
            DIAG["downstream_datasets"][label] = {
                "loaded": True, "n": int(len(ds)),
                "loader_idx_used": i, "columns": list(ds.column_names),
            }
            return ds
        except Exception as e:
            last_err = e
            continue
    print(f"  ✗ {label}: FAILED — {last_err}")
    DIAG["downstream_datasets"][label] = {"loaded": False, "error": str(last_err)}
    DIAG["errors"].append(f"dataset_load[{label}]: {last_err}")
    return None


# Subsample sizes scale with TARGET_SAMPLES so a smoke run (TARGET_SAMPLES=50)
# only evaluates a small fraction of each downstream dataset.
_scale = max(0.05, TARGET_SAMPLES / 1000.0)
def _sub(default_n):
    return max(20, int(round(default_n * _scale)))
SUBSAMPLES = {"truthfulqa":     None if _scale >= 0.5 else _sub(200),
              "triviaqa":       _sub(500),
              "coqa":           _sub(500),
              "tydiqa":         _sub(500),
              "halueval_qa":    _sub(1000),
              "halueval_summ":  _sub(500),
              "halueval_dialog": _sub(1000)}
print(f"Subsample scale = {_scale:.2f}; sizes = {SUBSAMPLES}")
DIAG["downstream_datasets"]["subsamples_requested"] = dict(SUBSAMPLES)

DATASETS = {}

DATASETS["truthfulqa"] = safe_load_first(
    lambda: load_dataset("truthfulqa/truthful_qa", "generation", split="validation"),
    lambda: load_dataset("truthful_qa", "generation", split="validation"),
    subsample=SUBSAMPLES["truthfulqa"], label="truthfulqa")

DATASETS["triviaqa"] = safe_load_first(
    lambda: load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation"),
    lambda: load_dataset("trivia_qa", "rc.nocontext", split="validation"),
    subsample=SUBSAMPLES["triviaqa"], label="triviaqa")

DATASETS["coqa"] = safe_load_first(
    lambda: load_dataset("stanfordnlp/coqa", split="validation"),
    subsample=SUBSAMPLES["coqa"], label="coqa")

def _load_tydiqa():
    ds = load_dataset("google-research-datasets/tydiqa", "secondary_task",
                       split="validation")
    return ds.filter(lambda x: "english" in x.get("id", "").lower())

def _load_tydiqa_legacy():
    ds = load_dataset("tydiqa", "secondary_task", split="validation")
    return ds.filter(lambda x: "english" in x.get("id", "").lower())

DATASETS["tydiqa"] = safe_load_first(
    _load_tydiqa, _load_tydiqa_legacy,
    subsample=SUBSAMPLES["tydiqa"], label="tydiqa")

def _load_halueval(config_name):
    for cand in ("data", "train", "validation"):
        try:
            return load_dataset("pminervini/HaluEval", config_name, split=cand)
        except (ValueError, KeyError, FileNotFoundError):
            continue
    dd = load_dataset("pminervini/HaluEval", config_name)
    return dd[list(dd.keys())[0]]

for cfg, label in [("qa","halueval_qa"), ("summarization","halueval_summ"),
                   ("dialogue","halueval_dialog")]:
    DATASETS[label] = safe_load_first(
        lambda c=cfg: _load_halueval(c),
        subsample=SUBSAMPLES[label], label=label)

print()
for k, v in DATASETS.items():
    print(f"  {k:18s} -> {len(v) if v else 'unavailable'}")


In [ ]:
# =============================================================================
# BLOCK 12: LIGHTWEIGHT GOLD-VS-GENERATION SCORER (sentence-transformers cosine)
# =============================================================================
from sentence_transformers import SentenceTransformer
import numpy as np

scorer = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                              device=str(device))

def score_match(gen: str, golds, threshold: float = 0.5) -> int:
    if not golds:
        return 0
    if isinstance(golds, str):
        golds = [golds]
    embs = scorer.encode([gen] + list(golds), convert_to_numpy=True,
                          normalize_embeddings=True)
    sims = embs[0] @ embs[1:].T
    return 0 if float(sims.max()) >= threshold else 1

print("✓ sentence-transformers scorer ready")


In [ ]:
# =============================================================================
# BLOCK 13: MULTI-TASK EVALUATION
# =============================================================================
@torch.no_grad()
def features_at_last_token(prompt: str, generation: str = ""):
    text = (prompt + " " + generation).strip()
    canon, D, V, H = extract_features(text)
    return canon, D, V, H


@torch.no_grad()
def generate_short_answer(prompt: str, max_new: int = MAX_GEN_NEW) -> str:
    _max_pos = getattr(model.config, "max_position_embeddings", 4096)
    enc = tokenizer(prompt, return_tensors="pt",
                    truncation=True,
                    max_length=max(_max_pos - max_new, 256)
                    ).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new,
                          do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc.input_ids.shape[1]:],
                            skip_special_tokens=True).strip()


def classify_features(canon, D, V, H):
    scalars = scaler.transform([[D, V, H]]).astype(np.float32)
    x = np.concatenate([np.array(canon, dtype=np.float32), scalars[0]])
    x = torch.from_numpy(x).unsqueeze(0).to(device)
    logits = mlp(x)
    p_hall = float(torch.softmax(logits, dim=1)[0, 1].item())
    pred = int(logits.argmax(1).item())
    return pred, p_hall


def eval_open_qa(ds, prompt_fn, gold_fn, n_max=None, dsname=""):
    preds, probs, golds = [], [], []
    failures = 0
    iterable = ds if n_max is None else ds.select(range(min(n_max, len(ds))))
    for s in tqdm(iterable, desc=dsname):
        try:
            prompt = prompt_fn(s)
            gen    = generate_short_answer(prompt, max_new=MAX_GEN_NEW)
            gold   = gold_fn(s)
            label  = score_match(gen, gold)
            canon, D, V, H = features_at_last_token(prompt, gen)
            pred, prob = classify_features(canon, D, V, H)
            preds.append(pred); probs.append(prob); golds.append(label)
        except Exception:
            failures += 1
            continue
    return np.array(golds), np.array(preds), np.array(probs), failures


def eval_halueval(ds, prompt_fn, right_key, wrong_key, n_max=None, dsname=""):
    preds, probs, golds = [], [], []
    failures = 0
    iterable = ds if n_max is None else ds.select(range(min(n_max, len(ds))))
    for s in tqdm(iterable, desc=dsname):
        try:
            prompt = prompt_fn(s)
            for ans_key, gold_label in [(right_key, 0), (wrong_key, 1)]:
                ans = s[ans_key]
                canon, D, V, H = features_at_last_token(prompt, ans)
                pred, prob = classify_features(canon, D, V, H)
                preds.append(pred); probs.append(prob); golds.append(gold_label)
        except Exception:
            failures += 1
            continue
    return np.array(golds), np.array(preds), np.array(probs), failures


def _free_cuda():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

def compute_metrics(y, p, pr):
    if len(y) == 0:
        return {"n": 0}
    acc  = float(accuracy_score(y, p))
    prec, rec, f1, _ = precision_recall_fscore_support(y, p, average="binary",
                                                         zero_division=0)
    try:    auc = float(roc_auc_score(y, pr)) if len(set(y.tolist())) > 1 else float("nan")
    except ValueError: auc = float("nan")
    try:    brier = float(brier_score_loss(y, pr))
    except ValueError: brier = float("nan")
    cm = confusion_matrix(y, p, labels=[0,1])
    return {"n": int(len(y)),
            "accuracy": acc, "precision": float(prec), "recall": float(rec),
            "f1": float(f1), "auc_roc": auc, "brier": brier,
            "cm": {"tn": int(cm[0,0]), "fp": int(cm[0,1]),
                   "fn": int(cm[1,0]), "tp": int(cm[1,1])},
            "label_distribution": {"y=0": int((y==0).sum()), "y=1": int((y==1).sum())},
            "pred_distribution":  {"p=0": int((p==0).sum()), "p=1": int((p==1).sum())},
            "prob_min":  round(float(pr.min()) if len(pr) else 0.0, 4),
            "prob_max":  round(float(pr.max()) if len(pr) else 0.0, 4),
            "prob_mean": round(float(pr.mean()) if len(pr) else 0.0, 4)}


def _store_raw(name, y, p, pr, failures):
    DIAG["downstream_datasets"][name] = DIAG["downstream_datasets"].get(name, {})
    DIAG["downstream_datasets"][name].update({
        "raw_y_pred_prob_first_50": [
            {"y": int(yy), "p": int(pp), "prob": round(float(prob), 4)}
            for yy, pp, prob in zip(y[:50], p[:50], pr[:50])
        ],
        "n_eval_failures": int(failures),
    })


multitask = {}
_t0 = time.perf_counter()

if DATASETS["truthfulqa"] is not None:
    y, p, pr, fails = eval_open_qa(DATASETS["truthfulqa"],
        prompt_fn=lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        gold_fn=lambda s: s.get("correct_answers", []),
        dsname="truthfulqa")
    multitask["truthfulqa"] = compute_metrics(y, p, pr); _store_raw("truthfulqa", y, p, pr, fails); _free_cuda()

if DATASETS["triviaqa"] is not None:
    y, p, pr, fails = eval_open_qa(DATASETS["triviaqa"],
        prompt_fn=lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        gold_fn=lambda s: list(s["answer"].get("aliases", [])) + [s["answer"].get("value", "")],
        dsname="triviaqa")
    multitask["triviaqa"] = compute_metrics(y, p, pr); _store_raw("triviaqa", y, p, pr, fails)

if DATASETS["coqa"] is not None:
    def coqa_first_turn(s):
        ctx = s["story"][:800]
        q = s["questions"][0] if s["questions"] else ""
        return f"Answer the question concisely based on the context: Context: {ctx} Q: {q} A:"
    def coqa_gold(s):
        a = s["answers"]
        return a["input_text"][0] if a["input_text"] else ""
    y, p, pr, fails = eval_open_qa(DATASETS["coqa"],
        prompt_fn=coqa_first_turn, gold_fn=coqa_gold, dsname="coqa")
    multitask["coqa"] = compute_metrics(y, p, pr); _store_raw("coqa", y, p, pr, fails)

if DATASETS["tydiqa"] is not None:
    y, p, pr, fails = eval_open_qa(DATASETS["tydiqa"],
        prompt_fn=lambda s: (
            f"Answer the question concisely based on the context: "
            f"Context: {s['context'][:800]} Q: {s['question']} A:"),
        gold_fn=lambda s: s.get("answers", {}).get("text", []),
        dsname="tydiqa")
    multitask["tydiqa"] = compute_metrics(y, p, pr); _store_raw("tydiqa", y, p, pr, fails)

if DATASETS["halueval_qa"] is not None:
    y, p, pr, fails = eval_halueval(DATASETS["halueval_qa"],
        prompt_fn=lambda s: (f"Answer the question concisely based on the context: "
                             f"Context: {s['knowledge'][:600]} Q: {s['question']} A:"),
        right_key="right_answer", wrong_key="hallucinated_answer",
        dsname="halueval_qa")
    multitask["halueval_qa"] = compute_metrics(y, p, pr); _store_raw("halueval_qa", y, p, pr, fails)

if DATASETS["halueval_summ"] is not None:
    y, p, pr, fails = eval_halueval(DATASETS["halueval_summ"],
        prompt_fn=lambda s: f"{s['document'][:800]} Please summarise the above article concisely. A:",
        right_key="right_summary", wrong_key="hallucinated_summary",
        dsname="halueval_summ")
    multitask["halueval_summ"] = compute_metrics(y, p, pr); _store_raw("halueval_summ", y, p, pr, fails)

if DATASETS["halueval_dialog"] is not None:
    y, p, pr, fails = eval_halueval(DATASETS["halueval_dialog"],
        prompt_fn=lambda s: (f"You are an assistant. Knowledge: {s['knowledge'][:400]}\n"
                             f"Dialogue: {s['dialogue_history'][:400]}\n[Assistant]:"),
        right_key="right_response", wrong_key="hallucinated_response",
        dsname="halueval_dialog")
    multitask["halueval_dialog"] = compute_metrics(y, p, pr); _store_raw("halueval_dialog", y, p, pr, fails)

DIAG["stage_timings_sec"]["multitask_eval"] = round(time.perf_counter() - _t0, 2)
DIAG["multitask"] = multitask
print("\n=== MULTI-TASK RESULTS ===")
for k, v in multitask.items():
    if v.get("n", 0) > 0:
        print(f"  {k:18s}  n={v['n']:5d}  Acc={v['accuracy']:.3f}  "
              f"F1={v['f1']:.3f}  AUROC={v['auc_roc']:.3f}")
print(f"\n✓ multitask_eval time: {DIAG['stage_timings_sec']['multitask_eval']} s")


In [ ]:
# =============================================================================
# BLOCK 14: FINAL RESULTS DUMP
# =============================================================================
DIAG["schema_version"] = "2.2-permodel"
DIAG["timestamp_utc"]  = (datetime.datetime.now(datetime.timezone.utc)
                             .replace(microsecond=0).isoformat().replace("+00:00", "Z"))
DIAG["host"] = ("kaggle" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ
                 else ("colab" if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
                       else "local"))
DIAG["features"] = [
    "canonical_mind (last token, last layer)",
    "D_mean (mean cosine distance between adjacent layers)",
    "V_last (L2 cross-layer variance of last-token activations)",
    "H_mean (mean Shannon entropy of per-step token distribution)",
]
DIAG["scaler"] = {
    "mean":  scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
    "features": ["D_mean", "V_last", "H_mean"],
}
DIAG["stage_timings_sec"]["total"] = round(sum(DIAG["stage_timings_sec"].values()), 2)

out_path = f"{MODEL_TAG}_results.json"
with open(out_path, "w") as f:
    json.dump(DIAG, f, indent=2, default=str)
print(f"\n✓ Wrote {out_path} ({os.path.getsize(out_path)/1024:.1f} KB)")

# Single-row summary table
print("\n" + "=" * 96)
print(f"FINAL TABLE — {MODEL_TAG}")
print("=" * 96)
print(f"{'Dataset':22s} {'N':>6s} {'Acc':>7s} {'Prec':>7s} {'Rec':>7s} {'F1':>7s} {'AUROC':>7s} {'Brier':>7s}")
print("-" * 96)
m = DIAG["wikipedia_eval"]
print(f"{'wikipedia (mind)':22s} {m['n']:6d} {m['accuracy']:7.3f} "
      f"{m['precision']:7.3f} {m['recall']:7.3f} {m['f1']:7.3f} "
      f"{m['auc_roc']:7.3f} {m['brier']:7.3f}")
for k, v in DIAG["multitask"].items():
    if v.get("n", 0) == 0: continue
    print(f"{k:22s} {v['n']:6d} {v['accuracy']:7.3f} "
          f"{v['precision']:7.3f} {v['recall']:7.3f} {v['f1']:7.3f} "
          f"{v['auc_roc']:7.3f} {v['brier']:7.3f}")
print("=" * 96)
print("\nTimings:")
for k, v in DIAG["stage_timings_sec"].items():
    print(f"  {k:22s} {v:>8.2f} s")
print(f"\nSkip counters (data-gen): {DIAG['data_gen'].get('skip_counters')}")
if DIAG["errors"]:
    print("\nNon-fatal errors captured:")
    for e in DIAG["errors"]:
        print(f"  - {e}")
print(f"\nPaste {out_path} back to the assistant for verification.")
print(f"Generated-data file: {MODEL_TAG}_dataset_full.json")
